# 실습 24. 민감도 분석

사회약학 연구방법 노트의 '실습 24 민감도 분석'에 나오는 셀을 차례로 모은 노트북입니다. 진행성 신세포암 1차 치료(신약 A 대 표준요법 B, 가상의 예시)의 분할생존모형에 일원 민감도 분석, 확률적 민감도 분석 5,000회, 비용효과 수용곡선을 직접 돌려 24장 본문의 숫자를 재현합니다. 자료 파일이 필요 없고 numpy, pandas, matplotlib, scipy만 씁니다. 셀을 위에서부터 차례로 실행하세요. 설명과 출력 읽는 법은 사이트의 실습 24 쪽에 있습니다.


## 가. 실습 준비

실습 23에서 만든 입력값 사전과 모형 함수 `run_model(p)`를 다시 적고, 곡선을 중앙값과 모양 모수로 읽고 바꾸는 함수를 만듭니다.

### 셀 1. 패키지와 입력값 사전

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

def weib_lam(median, gam):
    """중앙값(개월)과 모양 모수 -> lam. S(중앙값) = 0.5가 되게"""
    return np.log(2) / median ** gam

p = {
    # 표준요법 B의 와이블 곡선: 중앙 PFS 10개월, 중앙 OS 28개월
    "pfs_gam": 0.95, "pfs_lam": weib_lam(10, 0.95),
    "os_gam": 1.15, "os_lam": weib_lam(28, 1.15),
    # 신약 A의 위험비 (B 대비)
    "hr_pfs": 0.65, "hr_os": 0.75,
    # 상태별 효용, 이상반응의 QALY 손실 (1회)
    "u_pf": 0.78, "u_pd": 0.62,
    "du_ae_A": 0.012, "du_ae_B": 0.008,
    # 비용 (만원): 월 약값, 상태별 월 비용, 임종기와 이상반응 (1회)
    "c_drug_A": 190.0, "c_drug_B": 120.0,
    "c_pf": 40.0, "c_pd": 250.0, "c_death": 800.0,
    "c_ae_A": 120.0, "c_ae_B": 80.0,
    "disc": 0.045,                     # 연 할인율
}
print(len(p), "개 입력값")
print("pfs_lam =", round(p["pfs_lam"], 5),
      " os_lam =", round(p["os_lam"], 5))

### 셀 2. 모형 함수와 기준 분석

In [ ]:
def run_model(p, horizon=240):
    """입력값 사전 p -> 결과 사전 (시간 개월, 비용 만원)"""
    t = np.arange(0, horizon + 1)            # 주기 경계 시점
    disc = 1 / (1 + p["disc"]) ** ((t[:-1] + 0.5) / 12)
    res = {}
    for arm in ["A", "B"]:
        hr_pfs = p["hr_pfs"] if arm == "A" else 1.0
        hr_os = p["hr_os"] if arm == "A" else 1.0
        s_os = np.exp(-p["os_lam"] * hr_os * t ** p["os_gam"])
        s_pfs = np.exp(-p["pfs_lam"] * hr_pfs * t ** p["pfs_gam"])
        s_pfs = np.minimum(s_pfs, s_os)      # PFS가 OS를 넘지 않게
        pd_t = s_os - s_pfs                  # 진행 = OS - PFS
        pf = (s_pfs[:-1] + s_pfs[1:]) / 2    # 반주기 보정
        pd_ = (pd_t[:-1] + pd_t[1:]) / 2
        died = s_os[:-1] - s_os[1:]          # 주기별 사망
        q = (pf * p["u_pf"] + pd_ * p["u_pd"]) / 12
        r = {"ly_pf": pf.sum() / 12, "ly_pd": pd_.sum() / 12,
             "ly": (pf + pd_).sum() / 12,
             "ly_d": ((pf + pd_) * disc).sum() / 12,
             "qaly_undisc": q.sum() - p["du_ae_" + arm],
             "qaly": (q * disc).sum() - p["du_ae_" + arm],
             "c_drug": (pf * p["c_drug_" + arm] * disc).sum(),
             "c_pf": (pf * p["c_pf"] * disc).sum(),
             "c_pd": (pd_ * p["c_pd"] * disc).sum(),
             "c_death": (died * p["c_death"] * disc).sum(),
             "c_ae": p["c_ae_" + arm]}
        r["cost"] = (r["c_drug"] + r["c_pf"] + r["c_pd"]
                     + r["c_death"] + r["c_ae"])
        res[arm] = r
    res["d_cost"] = res["A"]["cost"] - res["B"]["cost"]
    res["d_qaly"] = res["A"]["qaly"] - res["B"]["qaly"]
    res["d_ly"] = res["A"]["ly_d"] - res["B"]["ly_d"]
    dq = res["d_qaly"]
    res["icer"] = res["d_cost"] / dq if dq != 0 else np.nan
    return res

base = run_model(p)
threshold = 5000                       # 이 예시에서 가정한 임계값
print("증분비용  :", round(base["d_cost"], 1))
print("증분 QALY :", round(base["d_qaly"], 4))
print("ICER      :", round(base["icer"]), "만원/QALY")
nmb = threshold * base["d_qaly"] - base["d_cost"]
print("증분 순금전편익:", round(nmb, 1), "만원")

### 셀 3. 곡선을 중앙값과 모양 모수로 읽고 바꾸는 함수

In [ ]:
def get_input(p, key):
    """입력값 표의 표현으로 읽기. pfs_med, os_med는 중앙값(개월)"""
    if key.endswith("_med"):
        c = key[:-4]                   # "pfs" 또는 "os"
        return (np.log(2) / p[c + "_lam"]) ** (1 / p[c + "_gam"])
    return p[key]

def set_input(p, key, value):
    """p의 사본에서 key 하나를 value로 바꾼다 (원래 p는 그대로)"""
    q = dict(p)
    if key.endswith("_med"):           # 중앙값을 바꾼다. 모양은 그대로
        c = key[:-4]
        q[c + "_lam"] = weib_lam(value, q[c + "_gam"])
    elif key.endswith("_gam"):         # 모양을 바꾼다. 중앙값은 그대로
        c = key[:-4]
        med = get_input(p, c + "_med")
        q[key] = value
        q[c + "_lam"] = weib_lam(med, value)
    else:                              # 그 밖의 입력값
        q[key] = value
    return q

print("중앙값:", round(get_input(p, "pfs_med"), 4),
      round(get_input(p, "os_med"), 4))
q1 = set_input(p, "os_gam", 1.29)      # 중앙값을 지키는 방법
q2 = dict(p, os_gam=1.29)              # 사전의 값만 바꾼 것
for name, q in [("set_input", q1), ("dict     ", q2)]:
    s60 = np.exp(-q["os_lam"] * 60 ** q["os_gam"])
    print(name, " os_lam", round(q["os_lam"], 5),
          " 중앙값", round(get_input(q, "os_med"), 1),
          " 60개월 생존율", round(s60, 3),
          " ICER", round(run_model(q)["icer"]))

## 나. 일원 민감도 분석과 토네이도 그림

입력값마다 범위를 정하고, 하나씩 양 끝으로 바꿔 ICER를 다시 계산합니다.

### 셀 4. 입력값의 분포와 적률법

In [ ]:
spec = {   # 이름: (분포, 불확실성의 크기)
    "hr_pfs": ("lognormal", 0.113), "hr_os": ("lognormal", 0.131),
    "pfs_med": ("lognormal", 0.07), "pfs_gam": ("lognormal", 0.05),
    "os_med": ("lognormal", 0.08), "os_gam": ("lognormal", 0.06),
    "u_pf": ("beta", 0.03), "u_pd": ("beta", 0.05),
    "du_ae_A": ("gamma", 0.25), "du_ae_B": ("gamma", 0.25),
    "c_pf": ("gamma", 0.20), "c_pd": ("gamma", 0.20),
    "c_death": ("gamma", 0.20),
    "c_ae_A": ("gamma", 0.25), "c_ae_B": ("gamma", 0.25),
}

def beta_ab(m, se):
    """평균, 표준오차 -> 베타분포의 alpha, beta"""
    n = m * (1 - m) / se ** 2 - 1
    return m * n, (1 - m) * n

def gamma_ks(m, se):
    """평균, 표준오차 -> 감마분포의 모양, 척도"""
    shape = (m / se) ** 2
    return shape, m / shape

print(len(spec), "개 입력값에 분포")
print("무진행 효용 0.78 (SE 0.03):", np.round(beta_ab(0.78, 0.03), 2))
print("진행 상태 비용 250 (SE 50):", gamma_ks(250, 250 * 0.20))

### 셀 5. 입력값별 범위 표

In [ ]:
z = stats.norm.ppf(0.975)              # 1.96

def range95(p, key):
    """분포의 2.5, 97.5 백분위수"""
    dist, u = spec[key]
    m = get_input(p, key)
    if dist == "lognormal":            # u = 로그 척도의 표준오차
        return np.exp(np.log(m) - z * u), np.exp(np.log(m) + z * u)
    if dist == "beta":                 # u = 표준오차
        a, b = beta_ab(m, u)
        return tuple(stats.beta.ppf([0.025, 0.975], a, b))
    k, s = gamma_ks(m, m * u)          # u = 표준오차 / 평균
    return tuple(stats.gamma.ppf([0.025, 0.975], k, scale=s))

rows = []
for key in spec:
    low, high = range95(p, key)
    rows.append([key, get_input(p, key), low, high, "95% 구간"])
a, b = p["c_drug_A"], p["c_drug_B"]
rows.append(["c_drug_A", a, a * (1 - 0.20), a, "20% 인하까지"])
rows.append(["c_drug_B", b, b * (1 - 0.20), b * (1 + 0.20),
             "±20% (임의)"])
ranges = pd.DataFrame(rows, columns=["key", "base", "low", "high",
                                     "basis"]).set_index("key")
ranges.round(3)

### 셀 6. 입력값을 하나씩 양 끝으로 바꿔 ICER 계산

In [ ]:
rows = []
for key in ranges.index:
    low, high = ranges.loc[key, "low"], ranges.loc[key, "high"]
    r_low = run_model(set_input(p, key, low))     # 낮은 값으로
    r_high = run_model(set_input(p, key, high))   # 높은 값으로
    rows.append([key, r_low["icer"], r_high["icer"]])
ow = pd.DataFrame(rows, columns=["key", "icer_low",
                                 "icer_high"]).set_index("key")
ow["swing"] = (ow["icer_high"] - ow["icer_low"]).abs()
lowest = ow[["icer_low", "icer_high"]].min(axis=1)
ow["below"] = lowest < threshold       # 임계값 아래로 내려가는가
ow = ow.sort_values("swing", ascending=False)
print("기준 분석 ICER:", round(base["icer"]))
ow.round(0)

### 셀 7. 토네이도 그림

In [ ]:
labels = {
    "hr_os": "HR, overall survival",
    "hr_pfs": "HR, progression-free survival",
    "os_med": "Median OS, therapy B",
    "os_gam": "Weibull shape, OS",
    "pfs_med": "Median PFS, therapy B",
    "pfs_gam": "Weibull shape, PFS",
    "u_pf": "Utility, progression-free",
    "u_pd": "Utility, progressed",
    "du_ae_A": "AE QALY loss, drug A",
    "du_ae_B": "AE QALY loss, therapy B",
    "c_drug_A": "Monthly price, drug A",
    "c_drug_B": "Monthly price, therapy B",
    "c_pf": "Monthly cost, progression-free",
    "c_pd": "Monthly cost, progressed",
    "c_death": "End-of-life cost",
    "c_ae_A": "AE cost, drug A",
    "c_ae_B": "AE cost, therapy B",
}
b0 = base["icer"]
y = np.arange(len(ow))                 # 0, 1, ..., 16
fig, ax = plt.subplots(figsize=(8, 5.6))
ax.barh(y, ow["icer_low"] - b0, left=b0, color="tab:blue",
        label="Lower value of input")
ax.barh(y, ow["icer_high"] - b0, left=b0, color="tab:orange",
        label="Upper value of input")
ax.axvline(b0, color="black", lw=1)
ax.axvline(threshold, color="gray", ls="--")
ax.set_yticks(y)
ax.set_yticklabels([labels[k] for k in ow.index])
ax.invert_yaxis()                      # 폭이 큰 것을 위에
ax.set_xlabel("ICER (10,000 KRW per QALY)")
ax.legend(loc="lower right")
plt.tight_layout()

### 셀 8. ICER가 임계값과 같아지는 약값

In [ ]:
prices = [152.0, 190.0]
icers = [run_model(dict(p, c_drug_A=x))["icer"] for x in prices]
print("ICER:", np.round(icers, 1))
price_star = np.interp(threshold, icers, prices)
print("임계 가격:", round(price_star, 2), "만원,",
      "인하율:", round((1 - price_star / 190) * 100, 1), "%")
r = run_model(dict(p, c_drug_A=price_star))
print("그 가격에서의 ICER:", round(r["icer"], 4))
slope = (icers[1] - icers[0]) / (prices[1] - prices[0])
print("약값 1만원당 ICER 변화:", round(slope, 1))

## 다. 확률적 민감도 분석

입력값을 분포에서 한꺼번에 뽑아 모형을 5,000번 돌립니다. 셀 12와 15는 몇 초 걸릴 수 있습니다.

### 셀 9. 상관 있는 두 정규 난수 만들기

In [ ]:
rng0 = np.random.default_rng(1)        # 연습용 난수 (본 분석과 별개)
z1 = rng0.normal(size=10000)           # 표준정규 난수 1만 개
e = rng0.normal(size=10000)            # z1과 독립인 난수
rho = 0.5
z2 = rho * z1 + np.sqrt(1 - rho ** 2) * e

print("z1과 e의 상관 :", round(np.corrcoef(z1, e)[0, 1], 3))
print("z1과 z2의 상관:", round(np.corrcoef(z1, z2)[0, 1], 3))
print("z2의 평균, 표준편차:", round(z2.mean(), 3), round(z2.std(), 3))
print("z1 > 0일 때 z2 > 0인 비율:", round(np.mean(z2[z1 > 0] > 0), 3))

fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.4), sharey=True)
axes[0].scatter(z1[:2000], e[:2000], s=3, alpha=0.3)
axes[0].set_title("Independent (rho = 0)")
axes[1].scatter(z1[:2000], z2[:2000], s=3, alpha=0.3)
axes[1].set_title("Correlated (rho = 0.5)")
for ax in axes:
    ax.set_xlabel("z1")
axes[0].set_ylabel("Second draw")
plt.tight_layout()

### 셀 10. 곡선과 위험비 한 벌 뽑기

In [ ]:
pairs = [("pfs_med", "os_med"),        # 함께 뽑는 짝 (PFS, OS)
         ("pfs_gam", "os_gam"),
         ("hr_pfs", "hr_os")]

def draw_curves(rng, p, rho=0.5):
    """곡선과 위험비 한 벌. 짝마다 로그 척도에서 상관 rho"""
    v = {}
    for k1, k2 in pairs:
        z1, z2 = rng.normal(size=2)    # 독립인 표준정규 난수 둘
        z2 = rho * z1 + np.sqrt(1 - rho ** 2) * z2
        v[k1] = get_input(p, k1) * np.exp(spec[k1][1] * z1)
        v[k2] = get_input(p, k2) * np.exp(spec[k2][1] * z2)
    q = dict(p)
    for k in ["hr_pfs", "hr_os", "pfs_gam", "os_gam"]:
        q[k] = v[k]
    q["pfs_lam"] = weib_lam(v["pfs_med"], v["pfs_gam"])
    q["os_lam"] = weib_lam(v["os_med"], v["os_gam"])
    return q

def crossed(p, horizon=240):
    """어느 군에서든 PFS 곡선이 OS 곡선을 넘으면 True"""
    t = np.arange(0, horizon + 1)
    for hr_pfs, hr_os in [(p["hr_pfs"], p["hr_os"]), (1.0, 1.0)]:
        s_pfs = np.exp(-p["pfs_lam"] * hr_pfs * t ** p["pfs_gam"])
        s_os = np.exp(-p["os_lam"] * hr_os * t ** p["os_gam"])
        if (s_pfs > s_os).any():
            return True
    return False

rng = np.random.default_rng(20261002)
q = draw_curves(rng, p)
keys = [k for pair in pairs for k in pair]
print(pd.DataFrame({"base": [get_input(p, k) for k in keys],
                    "draw": [get_input(q, k) for k in keys]},
                   index=keys).round(3))
print("곡선이 엇갈리는가:", crossed(q))

### 셀 11. 입력값 한 벌 뽑기와 다시 뽑기

In [ ]:
def draw_inputs(rng, p, count, rho=0.5, redraw=True):
    """입력값 한 벌을 분포에서 뽑는다 (약값과 할인율은 고정)"""
    while True:
        q = draw_curves(rng, p, rho)   # 1) 곡선과 위험비
        count["tries"] += 1
        if not crossed(q):
            break                      # 엇갈리지 않으면 채택
        count["crossed"] += 1
        if not redraw:
            break                      # 다시 뽑기를 끈 경우
    for key, (dist, u) in spec.items():    # 2) 나머지 입력값
        if dist == "beta":
            a, b = beta_ab(p[key], u)
            q[key] = rng.beta(a, b)
        elif dist == "gamma":
            k, s = gamma_ks(p[key], p[key] * u)
            q[key] = rng.gamma(k, s)
    return q                           # 로그정규는 1)에서 뽑았다

rng = np.random.default_rng(20261002)  # 같은 seed로 처음부터
count = {"tries": 0, "crossed": 0}
q = draw_inputs(rng, p, count)
show = ["u_pf", "u_pd", "c_pf", "c_pd", "c_death", "c_drug_A"]
print({k: round(float(q[k]), 3) for k in show})
r = run_model(q)
print("첫 벌: 증분비용", round(r["d_cost"]),
      " 증분 QALY", round(r["d_qaly"], 3), " ICER", round(r["icer"]))
print(count)

### 셀 12. 5,000벌 돌리기

In [ ]:
def run_psa(p, n=5000, seed=20261002, rho=0.5, redraw=True):
    """입력값 n벌을 뽑아 모형을 n번 돌린다"""
    rng = np.random.default_rng(seed)
    count = {"tries": 0, "crossed": 0}
    dc, dq = np.empty(n), np.empty(n)  # 결과를 담을 빈 배열
    for i in range(n):
        r = run_model(draw_inputs(rng, p, count, rho, redraw))
        dc[i], dq[i] = r["d_cost"], r["d_qaly"]
    return dc, dq, count

t0 = time.time()
dc, dq, count = run_psa(p)
print("걸린 시간(초):", round(time.time() - t0, 1))
print("뽑은 횟수", count["tries"], " 버린 횟수", count["crossed"],
      " 비율", round(count["crossed"] / count["tries"], 4))
print(dc.shape, dq.shape)
print("처음 세 벌의 증분비용:", dc[:3].round(1))
print("처음 세 벌의 증분 QALY:", dq[:3].round(4))

### 셀 13. 결과 요약

In [ ]:
inmb = threshold * dq - dc             # 벌마다의 증분 순금전편익
rows = []
for name, x in [("d_cost", dc), ("d_qaly", dq), ("inmb", inmb)]:
    lo, hi = np.percentile(x, [2.5, 97.5])
    rows.append([name, x.mean(), lo, hi])
summ = pd.DataFrame(rows, columns=["result", "mean", "p2.5",
                                   "p97.5"]).set_index("result")
print(summ.round(3))
print("ICER (평균 / 평균):", round(dc.mean() / dq.mean()))
p_ce5 = np.mean(inmb > 0)
print("비용효과적일 확률:", p_ce5,
      " 몬테카를로 표준오차:",
      round(np.sqrt(p_ce5 * (1 - p_ce5) / len(dc)), 4))

### 셀 14. 비용효과평면과 사분면

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.6))
ax.scatter(dq, dc, s=3, alpha=0.25, label="5,000 simulations")
x = np.array([-0.2, 1.6])
ax.plot(x, threshold * x, color="black", ls="--",
        label="Threshold: 5,000 per QALY")
ax.scatter(base["d_qaly"], base["d_cost"], color="red",
           marker="x", s=70, zorder=3, label="Base case")
ax.axhline(0, color="gray", lw=0.8)
ax.axvline(0, color="gray", lw=0.8)
ax.set_xlabel("Incremental QALYs")
ax.set_ylabel("Incremental cost (10,000 KRW)")
ax.legend(loc="upper left")
plt.tight_layout()

quad = {"NE": (dq > 0) & (dc > 0), "SE": (dq > 0) & (dc <= 0),
        "NW": (dq <= 0) & (dc > 0), "SW": (dq <= 0) & (dc <= 0)}
print({k: int(v.sum()) for k, v in quad.items()})
print("임계값 선 아래의 점:", int((inmb > 0).sum()))

### 셀 15. 상관과 다시 뽑기를 바꿔 보기

In [ ]:
rows = []
for rho, redraw in [(0.5, True), (0.0, True),
                    (0.5, False), (0.0, False)]:
    c2, q2, n2 = run_psa(p, rho=rho, redraw=redraw)
    rows.append([rho, redraw, n2["tries"], n2["crossed"],
                 n2["crossed"] / n2["tries"],
                 np.mean(threshold * q2 - c2 > 0)])
pd.DataFrame(rows, columns=["rho", "redraw", "tries", "crossed",
                            "share", "p_ce"]).round(4)

## 라. 비용효과 수용곡선

셀 12에서 얻은 5,000벌의 증분비용 `dc`와 증분 QALY `dq`를 임계값별로 요약합니다.

### 셀 16. 증분 순금전편익과 비용효과적일 확률

In [ ]:
first = pd.DataFrame({"d_qaly": dq[:5], "d_cost": dc[:5]},
                     index=[1, 2, 3, 4, 5])
first["inmb_5000"] = 5000 * first["d_qaly"] - first["d_cost"]
first["inmb_6000"] = 6000 * first["d_qaly"] - first["d_cost"]
print(first.round({"d_qaly": 3, "d_cost": 0, "inmb_5000": 0,
                   "inmb_6000": 0}))

inmb = 5000 * dq - dc                  # 5,000벌 모두
print("양수인 벌:", int((inmb > 0).sum()), "/", len(inmb),
      "=", np.mean(inmb > 0))

### 셀 17. 임계값별 확률과 EVPI

In [ ]:
def p_ce(dc, dq, lam):
    """임계값 lam에서 순금전편익이 양수인 벌의 비율"""
    return np.mean(lam * dq - dc > 0)

def evpi(dc, dq, lam):
    """1인당 완전정보의 기대가치 (대안이 둘일 때)"""
    nb = lam * dq - dc
    return np.maximum(nb, 0).mean() - max(nb.mean(), 0)

rows = []
for lam in [3000, 4000, 5000, 5500, 6000, 7000, 8000, 10000, 12000]:
    rows.append([lam, (lam * dq - dc).mean(),
                 p_ce(dc, dq, lam), evpi(dc, dq, lam)])
cols = ["threshold", "mean_inmb", "p_ce", "evpi"]
tab = pd.DataFrame(rows, columns=cols).set_index("threshold")
tab.round({"mean_inmb": 0, "p_ce": 4, "evpi": 1})

### 셀 18. 수용곡선 그리기

In [ ]:
lams = np.arange(0, 12001, 100)        # 0, 100, ..., 12000
curve = np.array([p_ce(dc, dq, lam) for lam in lams])
print(len(lams), "개 임계값. 양 끝의 확률:", curve[0], curve[-1])

fine = np.arange(0, 12001, 10)         # 10만원 간격
curve_f = np.array([p_ce(dc, dq, lam) for lam in fine])
lam50 = fine[np.argmax(curve_f >= 0.5)]
print("확률이 50%를 처음 넘는 임계값:", lam50)

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(lams, curve)
ax.axvline(threshold, color="gray", ls="--")
ax.axhline(0.5, color="gray", ls=":")
ax.set_xlabel("Threshold (10,000 KRW per QALY)")
ax.set_ylabel("Probability that drug A is cost-effective")
ax.set_ylim(0, 1)
plt.tight_layout()

### 셀 19. EVPI 풀어 보기

In [ ]:
pos = inmb[inmb > 0]                   # 신약 A가 더 나았던 벌
print("순금전편익의 평균:", round(inmb.mean(), 1))
print("양수인 비율:", np.mean(inmb > 0),
      " 양수인 벌의 평균:", round(pos.mean(), 1))
print("둘의 곱:", round(np.mean(inmb > 0) * pos.mean(), 1),
      " evpi():", round(evpi(dc, dq, 5000), 1))

ev = np.array([evpi(dc, dq, lam) for lam in fine])
print("EVPI가 가장 큰 임계값:", fine[np.argmax(ev)],
      " 그때의 EVPI:", round(ev.max(), 1))
print("환자 3,000명이면(억원):",
      round(evpi(dc, dq, 5000) * 3000 / 10000, 1))

## 과제 정답

**과제 1.** 처음 500벌, 1,000벌, 5,000벌만 썼을 때 임계값 5,000만원/QALY에서 비용효과적일 확률과 몬테카를로 표준오차를 구하고, seed를 1, 2, 3으로 바꿔 5,000벌씩 다시 돌려 확률을 구하세요.

### 셀 20. 과제 1 정답. 반복 횟수와 몬테카를로 오차

In [ ]:
ok = (5000 * dq - dc) > 0              # 벌마다 비용효과적인가
for n in [500, 1000, 5000]:
    pr = ok[:n].mean()                 # 처음 n벌만 쓴 확률
    se = np.sqrt(pr * (1 - pr) / n)    # 몬테카를로 표준오차
    print(n, " 확률", round(pr, 4), " 표준오차", round(se, 4),
          " 범위", round(pr - 1.96 * se, 3), "-",
          round(pr + 1.96 * se, 3))

for seed in [1, 2, 3]:
    c2, q2, n2 = run_psa(p, seed=seed)
    print("seed", seed, " 확률", np.mean(5000 * q2 - c2 > 0))

**과제 2.** 신약 A의 월 약값(190, 175, 160만원)과 무진행생존 위험비(95% 구간의 낮은 쪽 끝, 0.65, 높은 쪽 끝)를 함께 바꾼 ICER 표(3 × 3)를 만드세요.

### 셀 21. 과제 2 정답. 약값과 무진행생존 위험비를 함께 바꾸기

In [ ]:
hr_low = ranges.loc["hr_pfs", "low"]     # 95% 구간의 양 끝
hr_high = ranges.loc["hr_pfs", "high"]
hrs = [hr_low, 0.65, hr_high]
rows = []
for price in [190.0, 175.0, 160.0]:
    row = []
    for hr in hrs:
        q = set_input(set_input(p, "c_drug_A", price), "hr_pfs", hr)
        row.append(run_model(q)["icer"])
    rows.append(row)
two = pd.DataFrame(rows, index=[190, 175, 160],
                   columns=np.round(hrs, 2))
two.round(0)

**과제 3.** 신약 A의 월 약값을 임계 가격(셀 8의 `price_star`)과 160만원으로 낮춰 확률적 민감도 분석을 다시 돌리고, 임계값 5,000만원/QALY에서 비용효과적일 확률을 구해 세 가격의 수용곡선을 한 그림에 그리세요.

### 셀 22. 과제 3 정답. 약값을 낮췄을 때의 확률과 수용곡선

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
for price in [190.0, price_star, 160.0]:
    p2 = dict(p, c_drug_A=price)       # 약값만 바꾼 입력값
    c2, q2, n2 = run_psa(p2)           # 같은 seed로 5,000벌
    print("약값", round(price, 1),
          " ICER", round(run_model(p2)["icer"]),
          " 평균 순금전편익", round((5000 * q2 - c2).mean()),
          " 확률", p_ce(c2, q2, 5000),
          " QALY가 같은가", np.allclose(q2, dq))
    ax.plot(lams, [p_ce(c2, q2, lam) for lam in lams],
            label="Price " + str(round(price, 1)))
ax.axvline(threshold, color="gray", ls="--")
ax.axhline(0.5, color="gray", ls=":")
ax.set_xlabel("Threshold (10,000 KRW per QALY)")
ax.set_ylabel("Probability that drug A is cost-effective")
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()